In [1]:
"""
FINAL + simple state-machine gate ("3 strikes out / 2 hits back in").
Each expert (stock x cat) is ACTIVE initially.
 - Passes significance (layer1) to make a prediction at all.
 - ACTIVE: prediction is EXECUTED (counted). 3 CONSECUTIVE misses -> SUSPENDED.
 - SUSPENDED: prediction still evaluated silently (NOT counted). 2 CONSECUTIVE hits -> ACTIVE again.
Consecutive counters reset on the opposite outcome.
Baseline: expanding point-in-time up-rate. All 5 cats compete. n<10 flagged.
Reports NO-gate vs WITH-fsm side by side.
"""
import pandas as pd, numpy as np
from numpy.linalg import lstsq

TICKERS=['NVDA','MSFT','META','AMZN','GOOGL']
CATS=['chip','power','algorithm','regulation','earnings']
CONF_TH=0.6; BART_TH=0.6; SENT_MIN=0.05; SEED=323
PRED_START='2023-06-01'; K_LIST=[30,50]
STRIKES_OUT=3; HITS_IN=2; MIN_N=10

t2=pd.read_csv('data/table2_price_targets.csv'); t2['Date']=pd.to_datetime(t2['Date']).dt.normalize()
news=pd.read_csv('data/table1_news_scores.csv',encoding='utf-8-sig'); news['Date']=pd.to_datetime(news['Date']).dt.normalize()

def build(ticker):
    s=t2[t2.Ticker==ticker].sort_values('Date').reset_index(drop=True).copy()
    s['Daily_Return']=s['Close_t'].pct_change(); s['next_return']=s['ret_total']
    d=s[['Date','Daily_Return','next_return']].dropna().reset_index(drop=True)
    nk=news[news.Ticker==ticker]
    for cat in CATS:
        sub=nk[(nk.finbert_conf>=CONF_TH)&(nk[f'bart_{cat}']>=BART_TH)]
        a=sub.groupby('Date')['finbert_score'].mean().reset_index(); a.columns=['Date',f'{cat}_sent']
        d=d.merge(a,on='Date',how='left'); d[f'{cat}_sent']=d[f'{cat}_sent'].fillna(0)
    return d

def fit_beta(w):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); return lstsq(X,w[:,2],rcond=None)[0][0]
def is_sig(w,nb=400,seed=SEED):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); y=w[:,2]
    rng=np.random.default_rng(seed); bb=np.empty(nb)
    for i in range(nb):
        idx=rng.choice(len(X),len(X),replace=True); bb[i]=lstsq(X[idx],y[idx],rcond=None)[0][0]
    lo,hi=np.percentile(bb,5),np.percentile(bb,95); return (lo>0 or hi<0)

def collect(ticker,K,use_fsm):
    d=build(ticker)
    full=d[['Date','next_return']].dropna().sort_values('Date').reset_index(drop=True)
    fut=full['next_return'].values; fdate=full['Date'].values
    recs=[]
    for cat in CATS:
        col=f'{cat}_sent'
        dd=d[d[col].abs()>SENT_MIN][['Date',col,'Daily_Return','next_return']].dropna().reset_index(drop=True)
        arr=dd[[col,'Daily_Return','next_return']].values; dts=dd['Date'].values
        active=True; miss_streak=0; hit_streak=0
        for i in range(len(dd)):
            di=dts[i]
            if di<np.datetime64(PRED_START) or i<K: continue
            w=arr[i-K:i]
            if not is_sig(w): continue                 # layer1: only significant days predict
            beta=fit_beta(w); pred_up=(beta*arr[i,0])>0; act_up=arr[i,2]>0; hit=int(pred_up==act_up)
            if not use_fsm:
                past=fut[fdate<di]; base=(past>0).mean() if len(past)>20 else 0.5
                recs.append((pd.Timestamp(di).year,cat,hit,base)); continue
            # ---- FSM ----
            if active:
                past=fut[fdate<di]; base=(past>0).mean() if len(past)>20 else 0.5
                recs.append((pd.Timestamp(di).year,cat,hit,base))   # executed & counted
                if hit==0:
                    miss_streak+=1
                    if miss_streak>=STRIKES_OUT: active=False; miss_streak=0; hit_streak=0
                else:
                    miss_streak=0
            else:
                # suspended: silent, not counted, track consecutive hits to revive
                if hit==1:
                    hit_streak+=1
                    if hit_streak>=HITS_IN: active=True; hit_streak=0; miss_streak=0
                else:
                    hit_streak=0
    return pd.DataFrame(recs,columns=['year','cat','correct','base'])

for K in K_LIST:
    for use_fsm,tag in [(False,'NO gate'),(True,'WITH FSM (3 strikes out / 2 hits back, cold=active)')]:
        print(f"\n{'='*78}\nK={K} | {tag}\n{'='*78}")
        print(f"{'stock':6}{'cat':11}{'year':>6}{'acc':>8}{'base':>8}{'excess':>9}{'n':>6}  flag")
        for tk in TICKERS:
            df=collect(tk,K,use_fsm)
            if df.empty: print(f"{tk:6}(no predictions)"); print('-'*60); continue
            any_row=False
            for (cat,yr),sub in df.groupby(['cat','year']):
                n=len(sub)
                if n<5: continue
                acc=sub.correct.mean(); base=sub.base.mean(); flag='*small' if n<MIN_N else ''
                print(f"{tk:6}{cat:11}{yr:>6}{acc:>8.1%}{base:>8.1%}{acc-base:>+9.1%}{n:>6}  {flag}")
                any_row=True
            if not any_row: print(f"{tk:6}(no cat n>=5)")
            print('-'*60)

# ===================== OVERALL POOLED HIT-RATE =====================
print("\n\n"+"#"*78)
print("# OVERALL POOLED — every executed prediction across all stocks x cats")
print("#"*78)
for K in K_LIST:
    for use_fsm,tag in [(False,'NO gate'),(True,'WITH FSM (3-out/2-in)')]:
        allrec=[]
        for tk in TICKERS:
            df=collect(tk,K,use_fsm)
            if not df.empty: allrec.append(df)
        big=pd.concat(allrec,ignore_index=True)
        print(f"\n--- K={K} | {tag} ---")
        # by year
        for yr in sorted(big.year.unique()):
            sub=big[big.year==yr]
            acc=sub.correct.mean(); base=sub.base.mean()
            rng=np.random.default_rng(SEED)
            boot=np.array([rng.choice(sub.correct.values,len(sub),replace=True).mean() for _ in range(5000)])
            print(f"  {yr}: acc={acc:.1%}  base={base:.1%}  excess={acc-base:+.1%}  n={len(sub)}  "
                  f"P>base={(boot>base).mean():.0%}  P>50%={(boot>0.5).mean():.0%}")
        # all years
        acc=big.correct.mean(); base=big.base.mean()
        rng=np.random.default_rng(SEED)
        boot=np.array([rng.choice(big.correct.values,len(big),replace=True).mean() for _ in range(5000)])
        print(f"  ALL: acc={acc:.1%}  base={base:.1%}  excess={acc-base:+.1%}  n={len(big)}  "
              f"P>base={(boot>base).mean():.0%}  P>50%={(boot>0.5).mean():.0%}")


K=30 | NO gate
stock cat          year     acc    base   excess     n  flag
NVDA  algorithm    2024   42.1%   53.6%   -11.5%    19  
NVDA  chip         2024   33.3%   53.6%   -20.3%     6  *small
NVDA  chip         2025   64.7%   53.6%   +11.1%    51  
NVDA  power        2025   60.0%   53.7%    +6.3%     5  *small
NVDA  regulation   2025   61.5%   53.8%    +7.7%    13  
------------------------------------------------------------
MSFT  power        2025   66.7%   52.2%   +14.5%    30  
------------------------------------------------------------
META  earnings     2024   60.0%   51.6%    +8.4%     5  *small
META  earnings     2025   50.0%   51.8%    -1.8%    16  
META  power        2025   66.7%   51.7%   +15.0%    21  
------------------------------------------------------------
AMZN  power        2024   33.3%   51.0%   -17.6%    12  
------------------------------------------------------------
GOOGL algorithm    2025   40.0%   53.7%   -13.7%    20  
GOOGL earnings     2024   37.5%   

In [3]:
import pandas as pd, numpy as np
from numpy.linalg import lstsq
TICKERS=['NVDA','MSFT','META','AMZN','GOOGL']
CATS=['chip','power','algorithm','regulation','earnings']
CONF_TH=0.6; BART_TH=0.6; SENT_MIN=0.05; SEED=323
PRED_START='2023-06-01'; K_LIST=[30,50]
STRIKES_OUT=3; HITS_IN=2; MIN_N=10; REPORT_YEARS=[2024,2025]
t2=pd.read_csv('data/table2_price_targets.csv'); t2['Date']=pd.to_datetime(t2['Date']).dt.normalize()
news=pd.read_csv('data/table1_news_scores.csv',encoding='utf-8-sig'); news['Date']=pd.to_datetime(news['Date']).dt.normalize()
def build(ticker):
    s=t2[t2.Ticker==ticker].sort_values('Date').reset_index(drop=True).copy()
    s['Daily_Return']=s['Close_t'].pct_change(); s['next_return']=s['ret_total']
    d=s[['Date','Daily_Return','next_return']].dropna().reset_index(drop=True)
    nk=news[news.Ticker==ticker]
    for cat in CATS:
        sub=nk[(nk.finbert_conf>=CONF_TH)&(nk[f'bart_{cat}']>=BART_TH)]
        a=sub.groupby('Date')['finbert_score'].mean().reset_index(); a.columns=['Date',f'{cat}_sent']
        d=d.merge(a,on='Date',how='left'); d[f'{cat}_sent']=d[f'{cat}_sent'].fillna(0)
    return d
def fit_beta(w):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); return lstsq(X,w[:,2],rcond=None)[0][0]
def is_sig(w,nb=400,seed=SEED):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); y=w[:,2]
    rng=np.random.default_rng(seed); bb=np.empty(nb)
    for i in range(nb):
        idx=rng.choice(len(X),len(X),replace=True); bb[i]=lstsq(X[idx],y[idx],rcond=None)[0][0]
    lo,hi=np.percentile(bb,5),np.percentile(bb,95); return (lo>0 or hi<0)
def collect(ticker,K):
    d=build(ticker)
    full=d[['Date','next_return']].dropna().sort_values('Date').reset_index(drop=True)
    fut=full['next_return'].values; fdate=full['Date'].values
    recs=[]
    for cat in CATS:
        col=f'{cat}_sent'
        dd=d[d[col].abs()>SENT_MIN][['Date',col,'Daily_Return','next_return']].dropna().reset_index(drop=True)
        arr=dd[[col,'Daily_Return','next_return']].values; dts=dd['Date'].values
        active=True; miss_streak=0; hit_streak=0
        for i in range(len(dd)):
            di=dts[i]
            if di<np.datetime64(PRED_START) or i<K: continue
            w=arr[i-K:i]
            if not is_sig(w): continue
            beta=fit_beta(w); pred_up=(beta*arr[i,0])>0; act_up=arr[i,2]>0; hit=int(pred_up==act_up)
            if active:
                past=fut[fdate<di]; base=(past>0).mean() if len(past)>20 else 0.5
                recs.append((pd.Timestamp(di).year,cat,hit,base))
                if hit==0:
                    miss_streak+=1
                    if miss_streak>=STRIKES_OUT: active=False; miss_streak=0; hit_streak=0
                else: miss_streak=0
            else:
                if hit==1:
                    hit_streak+=1
                    if hit_streak>=HITS_IN: active=True; hit_streak=0; miss_streak=0
                else: hit_streak=0
    return pd.DataFrame(recs,columns=['year','cat','correct','base'])
for K in K_LIST:
    print(f"\n{'='*78}\nK={K} | WITH FSM\n{'='*78}")
    print(f"{'stock':6}{'cat':11}{'year':>6}{'acc':>8}{'base':>8}{'excess':>9}{'n':>6}  flag")
    for tk in TICKERS:
        df=collect(tk,K)
        if df.empty: print(f"{tk:6}(no predictions)"); print('-'*60); continue
        for (cat,yr),sub in df.groupby(['cat','year']):
            n=len(sub)
            if n<5: continue
            acc=sub.correct.mean(); base=sub.base.mean(); flag='*small' if n<MIN_N else ''
            print(f"{tk:6}{cat:11}{yr:>6}{acc:>8.1%}{base:>8.1%}{acc-base:>+9.1%}{n:>6}  {flag}")
        print('-'*60)
print("\n"+"#"*78+"\n# POOLED (2024 & 2025)\n"+"#"*78)
for K in K_LIST:
    allrec=[]
    for tk in TICKERS:
        df=collect(tk,K)
        if not df.empty: allrec.append(df)
    big=pd.concat(allrec,ignore_index=True)
    print(f"\n--- K={K} | WITH FSM ---")
    for yr in REPORT_YEARS:
        sub=big[big.year==yr]
        if sub.empty: continue
        acc=sub.correct.mean(); base=sub.base.mean()
        rng=np.random.default_rng(SEED)
        boot=np.array([rng.choice(sub.correct.values,len(sub),replace=True).mean() for _ in range(5000)])
        print(f"  {yr}: acc={acc:.1%}  base={base:.1%}  excess={acc-base:+.1%}  n={len(sub)}  P>base={(boot>base).mean():.0%}  P>50%={(boot>0.5).mean():.0%}")


K=30 | WITH FSM
stock cat          year     acc    base   excess     n  flag
NVDA  algorithm    2024   27.3%   53.6%   -26.4%    11  
NVDA  chip         2024   33.3%   53.6%   -20.3%     6  *small
NVDA  chip         2025   64.6%   53.6%   +11.0%    48  
NVDA  power        2025   60.0%   53.7%    +6.3%     5  *small
NVDA  regulation   2025   61.5%   53.8%    +7.7%    13  
------------------------------------------------------------
MSFT  power        2025   66.7%   52.2%   +14.5%    27  
------------------------------------------------------------
META  earnings     2024   60.0%   51.6%    +8.4%     5  *small
META  earnings     2025   53.8%   51.9%    +2.0%    13  
META  power        2025   66.7%   51.7%   +15.0%    21  
------------------------------------------------------------
AMZN  power        2024   42.9%   50.9%    -8.0%     7  *small
------------------------------------------------------------
GOOGL algorithm    2025   25.0%   53.8%   -28.8%    12  
GOOGL earnings     2024   3

In [4]:
"""
K=30 WITH FSM. PER-STOCK, BY-DAY voting.
For each stock separately:
  - same-day predictions across its categories are combined into ONE daily vote
    (majority: more up-votes -> stock predicts UP that day; ties dropped).
  - that daily vote is scored against the stock's own next-day direction.
  - per-stock, per-year direction accuracy vs the stock's own point-in-time base.
No cross-stock pooling here (equal-weight combination is a later step).
"""
import pandas as pd, numpy as np
from numpy.linalg import lstsq

TICKERS=['NVDA','MSFT','META','AMZN','GOOGL']
CATS=['chip','power','algorithm','regulation','earnings']
CONF_TH=0.6; BART_TH=0.6; SENT_MIN=0.05; SEED=323
PRED_START='2023-06-01'; K=30
STRIKES_OUT=3; HITS_IN=2; NBOOT=5000; REPORT_YEARS=[2024,2025]

t2=pd.read_csv('data/table2_price_targets.csv'); t2['Date']=pd.to_datetime(t2['Date']).dt.normalize()
news=pd.read_csv('data/table1_news_scores.csv',encoding='utf-8-sig'); news['Date']=pd.to_datetime(news['Date']).dt.normalize()

def build(ticker):
    s=t2[t2.Ticker==ticker].sort_values('Date').reset_index(drop=True).copy()
    s['Daily_Return']=s['Close_t'].pct_change(); s['next_return']=s['ret_total']
    d=s[['Date','Daily_Return','next_return']].dropna().reset_index(drop=True)
    nk=news[news.Ticker==ticker]
    for cat in CATS:
        sub=nk[(nk.finbert_conf>=CONF_TH)&(nk[f'bart_{cat}']>=BART_TH)]
        a=sub.groupby('Date')['finbert_score'].mean().reset_index(); a.columns=['Date',f'{cat}_sent']
        d=d.merge(a,on='Date',how='left'); d[f'{cat}_sent']=d[f'{cat}_sent'].fillna(0)
    return d

def fit_beta(w):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); return lstsq(X,w[:,2],rcond=None)[0][0]
def is_sig(w,nb=400,seed=SEED):
    X=np.column_stack([w[:,0],w[:,1],np.ones(len(w))]); y=w[:,2]
    rng=np.random.default_rng(seed); bb=np.empty(nb)
    for i in range(nb):
        idx=rng.choice(len(X),len(X),replace=True); bb[i]=lstsq(X[idx],y[idx],rcond=None)[0][0]
    lo,hi=np.percentile(bb,5),np.percentile(bb,95); return (lo>0 or hi<0)

def collect_votes(ticker):
    """Return one row per (date) where >=1 category made an executed prediction for this stock:
       date, year, up_votes, down_votes, act_up, next_return."""
    d=build(ticker)
    full=d[['Date','next_return']].dropna().sort_values('Date').reset_index(drop=True)
    fut=full['next_return'].values; fdate=full['Date'].values
    # per (date) accumulate votes from all categories
    votes={}   # date -> [up_count, down_count, act_up, year]
    for cat in CATS:
        col=f'{cat}_sent'
        dd=d[d[col].abs()>SENT_MIN][['Date',col,'Daily_Return','next_return']].dropna().reset_index(drop=True)
        arr=dd[[col,'Daily_Return','next_return']].values; dts=dd['Date'].values
        active=True; miss_streak=0; hit_streak=0
        for i in range(len(dd)):
            di=dts[i]
            if di<np.datetime64(PRED_START) or i<K: continue
            w=arr[i-K:i]
            if not is_sig(w): continue
            beta=fit_beta(w); pred_up=(beta*arr[i,0])>0; act_up=arr[i,2]>0; hit=int(pred_up==act_up)
            if active:
                di_ts=pd.Timestamp(di)
                if di_ts not in votes: votes[di_ts]=[0,0,act_up,di_ts.year]
                if pred_up: votes[di_ts][0]+=1
                else:       votes[di_ts][1]+=1
                if hit==0:
                    miss_streak+=1
                    if miss_streak>=STRIKES_OUT: active=False; miss_streak=0; hit_streak=0
                else: miss_streak=0
            else:
                if hit==1:
                    hit_streak+=1
                    if hit_streak>=HITS_IN: active=True; hit_streak=0; miss_streak=0
                else: hit_streak=0
    # combine votes into one daily decision (majority; ties dropped)
    rows=[]
    for di_ts,(up,down,act_up,yr) in votes.items():
        if up==down: continue                 # tie -> no decision this day, dropped
        day_pred_up = up>down
        day_hit = int(day_pred_up==act_up)
        rows.append((di_ts,yr,day_hit))
    vdf=pd.DataFrame(rows,columns=['date','year','correct']).sort_values('date').reset_index(drop=True)
    # per-stock point-in-time base: up-rate of this stock's own next_return before each voting day
    bases=[]
    for _,r in vdf.iterrows():
        past=fut[fdate<np.datetime64(r['date'])]
        bases.append((past>0).mean() if len(past)>20 else 0.5)
    vdf['base']=bases
    return vdf

# ===================== PER-STOCK, BY-DAY ACCURACY =====================
print(f"{'='*74}\nK={K} | WITH FSM | PER-STOCK, ONE VOTE PER DAY (majority across its cats)\n{'='*74}")
print(f"{'stock':6}{'year':>6}{'acc':>9}{'base':>8}{'excess':>9}{'n_days':>8}{'P>base':>9}{'P>50%':>8}")
for tk in TICKERS:
    vdf=collect_votes(tk)
    if vdf.empty: print(f"{tk:6}(no voting days)"); print('-'*60); continue
    for yr in REPORT_YEARS:
        sub=vdf[vdf.year==yr]
        if len(sub)<3: continue
        acc=sub.correct.mean(); base=sub.base.mean(); n=len(sub)
        rng=np.random.default_rng(SEED)
        boot=np.array([rng.choice(sub.correct.values,n,replace=True).mean() for _ in range(NBOOT)])
        print(f"{tk:6}{yr:>6}{acc:>9.1%}{base:>8.1%}{acc-base:>+9.1%}{n:>8}"
              f"{(boot>base).mean():>9.0%}{(boot>0.5).mean():>8.0%}")
    print('-'*60)

K=30 | WITH FSM | PER-STOCK, ONE VOTE PER DAY (majority across its cats)
stock   year      acc    base   excess  n_days   P>base   P>50%
NVDA    2024    29.4%   53.6%   -24.2%      17       1%      4%
NVDA    2025    63.3%   53.6%    +9.7%      49      91%     97%
------------------------------------------------------------
MSFT    2025    66.7%   52.2%   +14.5%      27      92%     96%
------------------------------------------------------------
META    2024    60.0%   51.6%    +8.4%       5      68%     68%
META    2025    64.5%   51.8%   +12.8%      31      91%     96%
------------------------------------------------------------
AMZN    2024    42.9%   50.9%    -8.0%       7      34%     34%
AMZN    2025    66.7%   50.9%   +15.7%       3      75%     75%
------------------------------------------------------------
GOOGL   2024    42.9%   53.6%   -10.7%      14      22%     22%
GOOGL   2025    40.0%   53.7%   -13.7%      25       9%     15%
-------------------------------------------

# 方法 / Method

全程 close-to-close。每个「股票 × 类别」是一个独立**专家**(expert),逐个信号日往前滚动预测次日涨跌。
All returns are close-to-close. Each `(stock, category)` pair is an independent **expert** that rolls forward one signal day at a time to predict next-day direction.

---

## 1. 滚动窗口 K / Rolling window K

**中文：** `K` 是回看的**信号日**数量(不是自然日,因为无新闻的日子已被剔除)。预测第 `i` 天时,只用它**之前**的 `K` 个信号日作训练窗口 `w = [i-K, i)`,严格不含当天及以后。每往前走一天,窗口整体平移一格、全部重新估计——即 walk-forward。`K=30`(约一个月信号)为主设定,对应「叙事按月演化」的假设;`K=50` 作稳健性对照。

**EN:** `K` is the number of **signal days** looked back (not calendar days — no-signal days are already removed). To predict day `i`, only the `K` signal days strictly before it form the training window `w = [i-K, i)`; nothing on or after `i` is used. The window shifts by one each day and is fully re-estimated — a walk-forward. `K=30` (≈ one month of signal) is primary, matching the "narratives evolve monthly" assumption; `K=50` is a robustness check.

---

## 2. 逐日滚动预测 / Rolling prediction per day

```mermaid
flowchart TD
    A["信号日 i / signal day i"] --> B{"i 早于 2023-06-01<br/>或前面信号日 < K?<br/>before 2023-06-01<br/>or fewer than K prior?"}
    B -- 是 yes --> Z["跳过 / skip"]
    B -- 否 no --> C["窗口 w = 过去 K 个信号日<br/>window = last K signal days"]
    C --> D["门1 显著性 / gate 1 significance"]
    D -- 不显著 not sig --> Z
    D -- 显著 sig --> E["拟合 beta / fit beta on w"]
    E --> F["预测方向 = sign(beta x 今日情绪)<br/>predict = sign(beta x today sent)"]
    F --> G["对比实际次日涨跌 -> 命中/未中<br/>vs actual -> hit / miss"]
    G --> H["门2 状态机 / gate 2 state machine"]
```

**中文：** 窗口够了,先过**门1**:在 `w` 上回归「情绪 + 当日收益 → 次日收益」,对情绪系数 `beta` bootstrap 400 次,只有 90% 区间整段不跨零(信号方向稳定)才准预测,否则闭嘴。过关后取 `beta`,预测方向 = `sign(beta × 今日情绪)`——历史学到的方向 × 今天情绪符号。与实际次日方向一致即命中。

**EN:** Once the window is full, pass **gate 1**: regress `next-return` on `[sentiment, same-day return]` over `w`, bootstrap the sentiment coefficient `beta` 400×, and predict only if its 90% interval is entirely one side of zero (stably signed); otherwise stay silent. If it passes, take `beta` and predict `sign(beta × today's sentiment)` — the learned direction times today's sentiment sign. A match with the actual next-day direction is a hit.

---

## 3. 门2 状态机 / Gate 2 state machine ("3 out / 2 in")

```mermaid
stateDiagram-v2
    [*] --> ACTIVE
    ACTIVE --> ACTIVE: 命中/hit — 计入/counted
    ACTIVE --> ACTIVE: 未中/miss, 连错<3 — 计入/counted
    ACTIVE --> SUSPENDED: 连错3次/3rd consecutive miss
    SUSPENDED --> SUSPENDED: 未中/miss — 不计入/not counted
    SUSPENDED --> SUSPENDED: 命中1次/1st hit — 不计入/not counted
    SUSPENDED --> ACTIVE: 连对2次/2nd consecutive hit
```

**中文：** 过门1让专家「能开口」,状态机决定「算不算数」。
- **ACTIVE(在岗):** 预测执行且**计入**统计;**连续错 3 次** → 停职。
- **SUSPENDED(停职):** 预测仍在算但**不计入**;**连续对 2 次** → 复职。
- 出现相反结果,连续计数清零。
两道门都只用**当时已揭晓**的过去结果,无未来泄漏。

**EN:** Gate 1 lets an expert *speak*; the state machine decides whether it *counts*.
- **ACTIVE:** predictions are executed and **counted**; 3 *consecutive* misses → suspend.
- **SUSPENDED:** predictions still evaluated but **not counted**; 2 *consecutive* hits → reactivate.
- Any opposite outcome resets the streak.
Both gates use only outcomes already revealed — no look-ahead leakage.

---

## 4. 按股票按天投票 / Per-stock, one vote per day

```mermaid
flowchart TD
    subgraph "同一股票, 同一天 / one stock, one day"
    C1["chip -> 涨/UP"]
    C2["power -> 涨/UP"]
    C3["regulation -> 跌/DOWN"]
    end
    C1 --> V["多数票, 平票丢弃<br/>majority, ties dropped"]
    C2 --> V
    C3 --> V
    V --> D["该股当天方向 / stock daily call"]
    D --> S["对比该股次日实际方向<br/>vs stock next-day actual"]
    S --> ACC["每股每年准确率 vs 该股自身基线<br/>per-stock per-year accuracy vs own baseline"]
```

**中文：** 同一股票、同一天若多个类别都在岗预测,取**多数票**合成「这只股票当天一票」(平票丢弃),再对比该股自身次日涨跌。每只股票各自按年统计方向准确率,与**该股自身**的 point-in-time 基线(该股次日收益在当天之前的历史上涨率)比较。**不做跨股等权重聚合**——等权重会让 GOOGL 的差与其他股票的好互相抵消,掩盖「信号有股票选择性」这一核心发现。

**EN:** If several categories of the same stock predict on the same day, a **majority vote** forms that stock's single daily call (ties dropped), scored against the stock's own next-day direction. Accuracy is computed per stock, per year, against that **stock's own** point-in-time baseline (its next-day up-rate over prior days). **No cross-stock equal-weight pooling** — equal weighting would cancel GOOGL's poor results against the others' strong ones and hide the key finding that the signal is stock-selective.

# 显著性 / Significance

## 结论 / Verdict

**中文：** **边缘显著(marginally significant),非标准强显著。** K=30、按股票按天投票、2025 样本外:

**EN:** **Marginally significant, not conventionally strong.** K=30, per-stock daily voting, 2025 out-of-sample:

| 股票 / Stock | 准确率 / Accuracy | 基线 / Baseline | 超额 / Excess | P>base |
|------|------|------|------|------|
| NVDA | 63.3% | 53.6% | +9.7% | 91% |
| MSFT | 66.7% | 52.2% | +14.5% | 92% |
| META | 64.5% | 51.8% | +12.8% | 91% |
| GOOGL | 40.0% | 53.7% | −13.7% | 9% |

**中文：** P>base ≈ 91–92% 对应单尾 p ≈ 0.08–0.09,落在 **10% 显著水平**,未达常规 5%。故为边缘显著。

**EN:** P>base ≈ 91–92% corresponds to a one-sided p ≈ 0.08–0.09 — the **10% level**, short of the conventional 5%. Hence marginally significant.

---

## 为什么只到边缘显著 / Why only marginal

**中文：**

1. **小样本(根本原因)。** 按天投票合成后,NVDA 一年仅 49 个投票日,MSFT 27,META 31。方向预测信噪比本就低(Tetlock 基准日度 R² 仅 1–3%),在数十个观测上要把单尾 p 压到 0.05 以下需准确率达 68%+。当前 63–67% 已属优秀,但样本量不足以跨过 5% 门槛——这是新闻数据量的物理上限,非方法缺陷。
2. **基线不低。** 2021–2025 为 AI 牛市,历史上涨率约 52–54%,故基线约 53%。需超越的是偏高的 53% 而非 50%,压缩了显著性余量。
3. **门在长窗口略削信号。** K=50 加门后 2025 的 P>base 由无门的 96% 降至 90%,状态机门在保护真信号的同时有轻微误伤。

**EN:**

1. **Small sample (root cause).** After daily voting, NVDA has only 49 voting days in a year, MSFT 27, META 31. Directional prediction is inherently low signal-to-noise (Tetlock's daily R² is only 1–3%); pushing a one-sided p below 0.05 on tens of observations needs ~68%+ accuracy. The observed 63–67% is strong, but the sample is too small to cross the 5% threshold — a physical limit of news volume, not a flaw in the method.
2. **Baseline is not low.** 2021–2025 is an AI bull market with a ~52–54% historical up-rate, so the baseline is ~53%. The target to beat is an elevated 53%, not 50%, which compresses the significance margin.
3. **The gate slightly weakens the signal at long windows.** With K=50, adding the gate drops 2025 P>base from 96% (no gate) to 90%; the state machine protects true signals but causes mild collateral loss.

---

## 为什么边缘显著在此并非弱点 / Why marginal is not a weakness here

**中文：** 证据的力量不在任何单行,而在其**结构**:

- **三家独立同向。** NVDA、MSFT、META 三只 AI 核心股**各自独立**达到 91–92%。三个独立检验同时超基线,这一**联合模式**偶然出现的概率远低于单个 10%。真正的证据是「三家一致为正、且 GOOGL 独立为负」,而非任一单股的边缘 P 值。
- **2024→2025 翻转。** 2024 这几家为负超额,2025 集体翻正。过拟合噪声不会产生「先失败后成功」的时间结构——若为过拟合,2024 也应「显著」,但并未。此跨年翻转排除了过拟合解释,是比单年 P 值更硬的证据。

**EN:** The strength of the evidence lies not in any single row but in its **structure**:

- **Three stocks, independently aligned.** NVDA, MSFT, and META each **independently** reach 91–92%. Three independent tests all exceeding baseline makes the **joint pattern** far less likely by chance than any single 10%. The real evidence is "three AI-core stocks jointly positive while GOOGL is independently negative," not any one stock's marginal p.
- **2024→2025 reversal.** These stocks show negative excess in 2024 and flip positive in 2025. Overfitting noise does not produce a "fail-then-succeed" time structure — if it were overfitting, 2024 should also look "significant," and it does not. This cross-year reversal rules out overfitting and is harder evidence than any single-year p-value.

